# Parallel Tool Calls — Batch Pattern

When a user query requires multiple independent lookups, sequential tool calls waste latency — each round trip (Claude → tool handler → Claude) adds delay before the final answer.

| Approach | Mechanism | Round trips | Latency |
|---|---|---|---|
| Sequential (default) | One tool per turn | N trips for N tools | High |
| **Batch pattern** | `batch_tool` wraps N calls in one turn | 1 trip | Low |

This notebook uses a knowledge-base lookup as the scenario: two independent tools (`get_domain_info` and `get_domain_weight`) answer different aspects of the same query — a natural candidate for parallel execution.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## Section 1: Sequential Tool Calls (Default Behavior)

We define two independent knowledge-base lookup tools:
- `get_domain_info` — returns the description of a given domain
- `get_domain_weight` — returns its recommended coverage percentage

Both answer the same user query but fetch **different, independent data** — a natural candidate for parallel execution.

First, we observe what Claude does *without* a batch tool present.

In [ ]:
// Mock handlers — return fixed data for demonstration
function getDomainInfo(domain: string): string {
  const info: Record<string, string> = {
    D1: 'Agentic Architecture & Orchestration — orchestrator/subagent design, agentic loops, hooks.',
    D2: 'Tool Design & MCP Integration — tool schema design, MCP primitives, transport selection.',
    D3: 'Claude Code Configuration & Workflows — CLAUDE.md hierarchy, custom commands.',
    D4: 'Prompt Engineering & Structured Output — JSON mode, null handling, format normalization.',
    D5: 'Context Management & Reliability — prompt caching, batch API, context compaction.',
  };
  return info[domain] ?? `No info found for domain: ${domain}`;
}

function getDomainWeight(domain: string): string {
  const weights: Record<string, string> = {
    D1: 'D1 covers ~25% of the curriculum — the highest-coverage topic.',
    D2: 'D2 covers ~20% of the curriculum.',
    D3: 'D3 covers ~20% of the curriculum.',
    D4: 'D4 covers ~20% of the curriculum.',
    D5: 'D5 covers ~15% of the curriculum.',
  };
  return weights[domain] ?? `No weight data for domain: ${domain}`;
}

function processToolCall(toolName: string, toolInput: Record<string, string>): string {
  if (toolName === 'get_domain_info') return getDomainInfo(toolInput.domain);
  if (toolName === 'get_domain_weight') return getDomainWeight(toolInput.domain);
  throw new Error(`Unknown tool: ${toolName}`);
}

const domainInfoTool: Anthropic.Tool = {
  name: 'get_domain_info',
  description: 'Returns the description of a knowledge domain.',
  input_schema: {
    type: 'object',
    properties: {
      domain: { type: 'string', description: 'Domain code: D1, D2, D3, D4, or D5' },
    },
    required: ['domain'],
  },
};

const domainWeightTool: Anthropic.Tool = {
  name: 'get_domain_weight',
  description: 'Returns the recommended coverage percentage for a knowledge domain.',
  input_schema: {
    type: 'object',
    properties: {
      domain: { type: 'string', description: 'Domain code: D1, D2, D3, D4, or D5' },
    },
    required: ['domain'],
  },
};

console.log('Tools defined: get_domain_info, get_domain_weight');

In [ ]:
// Reusable agentic loop — works for both sequential and batch demos
async function runAgentLoop(
  userQuery: string,
  tools: Anthropic.Tool[],
  processToolFn: (name: string, input: Record<string, unknown>) => string
): Promise<void> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: userQuery }];
  let roundTrips = 0;

  while (true) {
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 1024,
      tools,
      tool_choice: { type: 'auto' },
      messages,
    });

    roundTrips++;
    console.log(`--- Round trip ${roundTrips} (stop_reason: ${response.stop_reason}) ---`);

    for (const block of response.content) {
      if (block.type === 'text') console.log('  Claude:', block.text.slice(0, 150));
      if (block.type === 'tool_use') {
        console.log(`  Tool call: ${block.name}(${JSON.stringify(block.input)})`);
      }
    }

    if (response.stop_reason === 'end_turn') break;

    messages.push({ role: 'assistant', content: response.content });

    const toolResults: Anthropic.ToolResultBlockParam[] = response.content
      .filter((b): b is Anthropic.ToolUseBlock => b.type === 'tool_use')
      .map(b => ({
        type: 'tool_result' as const,
        tool_use_id: b.id,
        content: processToolFn(b.name, b.input as Record<string, unknown>),
      }));

    messages.push({ role: 'user', content: toolResults });
  }

  console.log(`Total round trips: ${roundTrips}`);
}

console.log('runAgentLoop helper defined.');

In [ ]:
const QUERY = 'Tell me about domain D2 — what is it, and how much coverage does it get?';

console.log('=== WITHOUT batch_tool: sequential tool calls ===');
console.log('Query:', QUERY);
console.log('');

await runAgentLoop(
  QUERY,
  [domainInfoTool, domainWeightTool],
  (name, input) => processToolCall(name, input as Record<string, string>)
);

## The Problem: N Independent Tools → N Round Trips

In the output above, Claude made **two separate round trips** even though both tools answer independent questions:

```
Round trip 1 → get_domain_info(D2)   → result returned
Round trip 2 → get_domain_weight(D2) → result returned
Round trip 3 → end_turn (final answer)
```

This is wasteful. In production:
- Each tool call adds latency (network I/O, processing)
- 5 independent lookups × 200 ms each = 1 s of unnecessary serial delay
- Under load, sequential tool calls become a bottleneck

**Root cause:** Claude Sonnet may not automatically parallelize tool calls even when `disable_parallel_tool_use` is not set.

**Fix:** Introduce a `batch_tool` meta-tool that signals Claude to bundle independent calls.

## Section 2: Batch Pattern — Parallel Execution in One Round Trip

The `batch_tool` is a **meta-tool**: Claude calls it with an array of `invocations`, and the handler executes all sub-calls before returning a combined result.

```
// batch_tool input shape
{
  invocations: [
    { name: 'get_domain_info',   arguments: '{"domain":"D2"}' },
    { name: 'get_domain_weight', arguments: '{"domain":"D2"}' }
  ]
}
```

Key design choices:
- `arguments` is a **JSON-stringified string** — the batch_tool schema is agnostic to what tools it wraps; inner schemas are resolved only when we deserialize and dispatch
- We execute invocations and join their results before returning a single `tool_result`
- When `batch_tool` is present, Claude uses it to combine parallel calls automatically

In [ ]:
const batchTool: Anthropic.Tool = {
  name: 'batch_tool',
  description: 'Invoke multiple other tools simultaneously in a single call.',
  input_schema: {
    type: 'object',
    properties: {
      invocations: {
        type: 'array',
        description: 'Array of tool calls to execute in parallel.',
        items: {
          type: 'object',
          properties: {
            name: { type: 'string', description: 'Name of the tool to invoke.' },
            arguments: { type: 'string', description: 'JSON-stringified arguments for the tool.' },
          },
          required: ['name', 'arguments'],
        },
      },
    },
    required: ['invocations'],
  },
};

function processToolCallWithBatch(
  toolName: string,
  toolInput: Record<string, unknown>
): string {
  if (toolName === 'batch_tool') {
    const invocations = toolInput.invocations as Array<{ name: string; arguments: string }>;
    return invocations
      .map(inv => processToolCall(inv.name, JSON.parse(inv.arguments) as Record<string, string>))
      .join('\n');
  }
  return processToolCall(toolName, toolInput as Record<string, string>);
}

console.log('batch_tool and processToolCallWithBatch defined.');

In [ ]:
console.log('=== WITH batch_tool: parallel tool calls ===');
console.log('Query:', QUERY);
console.log('');

await runAgentLoop(
  QUERY,
  [domainInfoTool, domainWeightTool, batchTool],
  processToolCallWithBatch
);

## Summary

| | Without `batch_tool` | With `batch_tool` |
|---|---|---|
| Tool calls per query | 1 per round trip | All at once |
| Round trips | N (one per tool) | 1 |
| Latency | N × tool_latency | ~1 × tool_latency |

### Key Concepts

**When to use the batch pattern:**
- Tools answer independent questions about the same entity
- No data dependency between tools (Tool B does not need Tool A's output as input)
- Latency reduction is a production requirement

**When NOT to use the batch pattern:**
- Tools are sequential by nature: search → retrieve → summarize
- Tool B's input depends on Tool A's output

**`tool_choice` interaction:**
- `{ type: 'auto' }` — Claude decides whether to use `batch_tool`
- `{ type: 'any' }` — Claude must call *some* tool (may pick `batch_tool`)
- `{ type: 'tool', name: 'batch_tool' }` — forces the batch_tool call explicitly

**Schema design note — `arguments` as a JSON string:**

`batch_tool` stores each sub-call's arguments as a stringified JSON string rather than a nested object. This makes `batch_tool` agnostic to the schemas of the tools it wraps — inner schemas are resolved only when we `JSON.parse` and dispatch each invocation.